# 🌐 Assistente Diversa — Educação Inclusiva
## Protótipo funcional para TCC

Este notebook entrega o protótipo funcional do TCC: um chat de Educação Inclusiva que usa uma base de conhecimento do Portal Diversa, busca por relevância com TF-IDF e geração de resposta com Llama 3.1 via Groq.

**Importante:** se você não informar a chave da Groq, o protótipo ainda funciona em modo demonstração, respondendo com base nos trechos recuperados da base.

## 1. Instalação de dependências



In [ ]:
%pip install -U beautifulsoup4 gradio pandas scikit-learn requests

print("✅ Instalação concluída!")
print()
print("Bibliotecas usadas no projeto:")
print("  • gradio     — interface web do chatbot")
print("  • requests   — chamadas HTTP para a Groq")
print("  • pandas     — manipulação da base de dados")
print("  • sklearn    — TF-IDF e busca por similaridade")


## 2. Importações e configuração



In [ ]:
import json
import os
import re
import time
from typing import Dict, List

import gradio as gr
import pandas as pd
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from dotenv import load_dotenv

# ============================================================
# CONFIGURAÇÃO DA GROQ
# ============================================================
# Por segurança, a chave NÃO deve ficar escrita diretamente no notebook.
# Opção recomendada no Google Colab:
# 1) Abra o menu "Secrets" (ícone de chave na lateral esquerda)
# 2) Crie um segredo chamado GROQ_KEY
# 3) Cole sua chave lá
#
# Sem chave, o projeto continua funcionando em modo demonstração,
# respondendo com base nos trechos recuperados da base.

load_dotenv()
GROQ_KEY = os.getenv("GROQ_KEY", "")

GROQ_MODEL = "llama-3.1-8b-instant"
GROQ_URL = "https://api.groq.com/openai/v1/chat/completions"

PARAMS_GERACAO = {
    "model": GROQ_MODEL,
    "max_tokens": 600,
    "temperature": 0.3,
    "top_p": 0.9,
}

print("✅ Configuração carregada")
print("Modelo:", GROQ_MODEL)
print("Modo:", "Groq/Llama" if GROQ_KEY else "Demonstração sem API")

## 3. Base de conhecimento


In [ ]:
import pandas as pd

sheet_id = "1Ll0gKNxMs20rMAESmjk4Dk2_FAtugKgY5NUhwaod1rI"
url_planilha = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

# 1. Transforma a planilha em um Pandas DataFrame
artigos_brutos = pd.read_csv(url_planilha)
artigos_brutos = artigos_brutos.rename(columns={"t": "titulo"})

# 2. Visualizando dimensões do Dataframe
print("Linhas x Colunas =", artigos_brutos.shape)


# 3. Imprime as colunas (corrigido: sem os parênteses no final)
print("\nColunas presentes na base de dados:")
print(list(artigos_brutos.columns))
print()

# 4. Visualiza as 5 primeiras linhas de forma formatada no Jupyter
print("5 primeiras linhas")
display(artigos_brutos.head())

# 5. Visualização dos tipos dos objetos
print(f"Todos os objetos são de texto:")
display(artigos_brutos.dtypes)

## 4. Limpeza da base



In [ ]:
def limpar_texto(texto):
    texto = re.sub(r"\s+", " ", str(texto))
    texto = re.sub(r"\[.*?\]", "", texto)
    texto = re.sub(r"http\S+", "", texto)
    return texto.strip()

df = pd.DataFrame(artigos_brutos)
df["texto_limpo"] = df["texto"].apply(limpar_texto)
df = df[df["texto_limpo"].str.len() > 120].reset_index(drop=True)
df["conteudo_busca"] = df["titulo"] + ". " + df["titulo"] + ". " + df["texto_limpo"]

print("✅ Limpeza concluída")
print("Total de artigos válidos:", len(df))
display(df[["titulo", "url"]])

## 5. Vetorização TF-IDF e busca semântica


In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=8000,
    sublinear_tf=True,
    strip_accents="unicode",
    lowercase=True,
)

matriz_tfidf = vectorizer.fit_transform(df["conteudo_busca"])

def buscar_artigos_relevantes(pergunta, top_k=3, score_minimo=0.015):
    vetor = vectorizer.transform([pergunta])
    scores = cosine_similarity(vetor, matriz_tfidf)[0]
    indices = scores.argsort()[::-1][:top_k]

    resultados = []
    for i in indices:
        score = float(scores[i])
        if score >= score_minimo:
            resultados.append({
                "titulo": df.iloc[i]["titulo"],
                "url": df.iloc[i]["url"],
                "trecho": df.iloc[i]["texto_limpo"][:900],
                "score": round(score, 4),
            })
    return resultados

print("✅ Vetorização concluída")
print("Matriz:", matriz_tfidf.shape)

## 6. Teste da busca

In [ ]:
pergunta_teste = "O que é Atendimento Educacional Especializado?"
resultados = buscar_artigos_relevantes(pergunta_teste)

print("Pergunta:", pergunta_teste)
print("Artigos recuperados:")
for r in resultados:
        print(f"- {r['titulo']} | score={r['score']}")

## 7. Prompt do assistente e controle de escopo


In [ ]:
SYSTEM_PROMPT = (
"Você é um assistente especialista em Educação Inclusiva do Portal Diversa (diversa.org.br)."
"Seu papel é apoiar professores, gestores escolares, familiares de pessoas com deficiência, estudantes, "
        "pesquisadores e profissionais de secretarias de educação, usando os TRECHOS fornecidos."
        "RESPONDA apenas temas de educação inclusiva: autismo (TEA), TDAH, deficiência visual, auditiva, intelectual "
        "e física, AEE, sala de recursos multifuncionais, tecnologia assistiva, acessibilidade, adaptações curriculares, "
        "DUA, legislação educacional, família, gestão escolar e conteúdos do Portal Diversa."
        "NÃO RESPONDA: diagnóstico médico ou psicológico, recomendação de medicamento/tratamento, opinião política "
        "partidária, decisão judicial individual, nem assuntos sem relação com educação inclusiva."
        "Nunca invente URLs, títulos, leis, números ou fontes. Cite apenas fontes presentes nos TRECHOS fornecidos. "
        "Se a base não tiver informação suficiente, diga isso claramente."
        "Responda sempre em português brasileiro, com linguagem clara, acessível, empática e prática."
    )

PALAVRAS_ESCOPO = [
        "educação", "inclus", "deficiência", "deficiente", "autismo", "tea", "tdah", "aee", "lbi", "bncc",
        "ldb", "aluno", "estudante", "escola", "professor", "gestor", "família", "sala de recursos",
        "tecnologia assistiva", "acessibilidade", "libras", "braille", "curricular", "pedagóg", "aprendizagem",
        "apoio escolar", "laudo", "diagnóstico", "barreira", "adaptação", "desenho universal", "dua"
    ]

def pergunta_dentro_escopo(pergunta):
        pergunta_normalizada = limpar_texto(pergunta).lower()
        return any(palavra in pergunta_normalizada for palavra in PALAVRAS_ESCOPO)

print("✅ Prompt configurado")

## 8. Funções de resposta



In [ ]:
# ============================================================
# PERFIS DE USUÁRIO
# ============================================================
# Cada perfil muda o comportamento da resposta gerada pela IA.
# Professor: resposta mais pedagógica e detalhada.
# Família: linguagem mais simples e acolhedora.
# Gestor: resposta mais técnica, objetiva e direta.

PERFIS_USUARIO = {
    "Professor": {"temperature": 0.4, "max_tokens": 600, "frequency_penalty": 0.3},
    "Família":   {"temperature": 0.5, "max_tokens": 350, "frequency_penalty": 0.2},
    "Gestor":    {"temperature": 0.1, "max_tokens": 300, "frequency_penalty": 0.0},
}

INSTRUCOES_PERFIL = {
    "Professor": (
        "Perfil selecionado: Professor. Responda de forma pedagógica, detalhada e aplicável em sala de aula. "
        "Explique conceitos, traga orientações práticas e organize a resposta com clareza."
    ),
    "Família": (
        "Perfil selecionado: Família. Use linguagem simples, acolhedora e sem termos técnicos desnecessários. "
        "Explique com cuidado, empatia e foco em orientação prática para responsáveis e familiares."
    ),
    "Gestor": (
        "Perfil selecionado: Gestor. Responda de forma técnica, objetiva e estratégica. "
        "Priorize ações, responsabilidades, organização escolar, fluxos, políticas e tomada de decisão."
    ),
}


def normalizar_perfil(perfil_usuario):
    """Garante que sempre exista um perfil válido selecionado."""
    if perfil_usuario in PERFIS_USUARIO:
        return perfil_usuario
    return "Professor"


def chamar_groq(messages, perfil_usuario="Professor", tentativas=3):
    if not GROQ_KEY:
        return ""

    perfil_usuario = normalizar_perfil(perfil_usuario)
    parametros_perfil = PERFIS_USUARIO[perfil_usuario]

    headers = {
        "Authorization": f"Bearer {GROQ_KEY}",
        "Content-Type": "application/json",
    }

    # Os parâmetros do perfil sobrescrevem os parâmetros gerais.
    payload = {
        **PARAMS_GERACAO,
        **parametros_perfil,
        "messages": messages,
    }

    for tentativa in range(1, tentativas + 1):
        try:
            resp = requests.post(GROQ_URL, headers=headers, json=payload, timeout=30)

            if resp.status_code == 200:
                try:
                    dados = resp.json()
                    return dados["choices"][0]["message"]["content"].strip()
                except Exception as exc:
                    return (
                        "❌ A Groq respondeu, mas o retorno não veio no formato esperado. "
                        f"Detalhe técnico: {exc}"
                    )

            if resp.status_code == 429:
                time.sleep(5 * tentativa)
                continue

            if resp.status_code == 401:
                return "❌ Chave da Groq inválida. Verifique a variável GROQ_KEY."

            return f"❌ Erro HTTP {resp.status_code}: {resp.text[:250]}"

        except Exception as exc:
            return f"❌ Erro inesperado ao chamar a Groq: {exc}"

    return "⏳ Serviço indisponível no momento."


def resposta_demo_sem_llm(pergunta, artigos, perfil_usuario="Professor"):
    perfil_usuario = normalizar_perfil(perfil_usuario)

    if not artigos:
        return (
            "Minha especialidade é Educação Inclusiva. A base atual ainda não possui trechos suficientes "
            "para responder essa pergunta com segurança. Recomendo ampliar a base com mais artigos do Portal Diversa."
        )

    principal = artigos[0]

    if perfil_usuario == "Família":
        introducao = (
            "**Resposta em modo demonstração, sem API da Groq configurada:**\n\n"
            f"Pelo artigo **{principal['titulo']}**, a ideia principal é acolher o estudante e observar quais barreiras "
            "podem estar dificultando sua participação na escola. A família pode conversar com a equipe escolar, "
            "compartilhar informações importantes e acompanhar os apoios combinados.\n\n"
        )
    elif perfil_usuario == "Gestor":
        introducao = (
            "**Resposta em modo demonstração, sem API da Groq configurada:**\n\n"
            f"Com base no artigo **{principal['titulo']}**, a gestão deve mapear barreiras, organizar apoios pedagógicos, "
            "articular professores e AEE, acompanhar registros e garantir acessibilidade nas práticas escolares.\n\n"
        )
    else:
        introducao = (
            "**Resposta em modo demonstração, sem API da Groq configurada:**\n\n"
            f"Com base no artigo **{principal['titulo']}**, a orientação principal é considerar a educação inclusiva "
            "como um processo de remoção de barreiras para que todos os estudantes participem da vida escolar. "
            "A escola deve planejar estratégias pedagógicas acessíveis, articular o Atendimento Educacional Especializado "
            "quando necessário e envolver professores, gestão, família e comunidade.\n\n"
        )

    return (
        introducao
        + "Trecho recuperado da base:\n"
        + f"> {principal['trecho'][:550]}...\n\n"
        + "Para respostas mais completas geradas por IA, configure a variável `GROQ_KEY`."
    )


def montar_fontes(artigos):
    if not artigos:
        return ""
    linhas = ["\n\n---", "📚 **Fontes recuperadas da base:**"]
    for artigo in artigos:
        linhas.append(f"- [{artigo['titulo']}]({artigo['url']}) — score {artigo['score']}")
    return "\n".join(linhas)


def responder(pergunta, historico=None, perfil_usuario="Professor"):
    pergunta = (pergunta or "").strip()
    perfil_usuario = normalizar_perfil(perfil_usuario)

    if not pergunta:
        return "Digite uma pergunta sobre Educação Inclusiva."

    artigos = buscar_artigos_relevantes(pergunta, top_k=3)

    if not pergunta_dentro_escopo(pergunta) and not artigos:
        return "Minha especialidade é Educação Inclusiva. Posso te ajudar com alguma dúvida sobre esse tema?"

    contexto = "\n\n".join(
        f"Artigo: {a['titulo']}\nURL: {a['url']}\nTrecho: {a['trecho']}\nScore: {a['score']}"
        for a in artigos
    ) or "Nenhum trecho relevante foi encontrado na base atual."

    instrucao_usuario = (
        "Use os TRECHOS DO PORTAL DIVERSA abaixo para responder. "
        "Não invente fontes. Caso a informação não esteja nos trechos, diga que a base atual é insuficiente.\n\n"
        f"TRECHOS DO PORTAL DIVERSA:\n{contexto}\n\n"
        f"PERGUNTA DO USUÁRIO: {pergunta}"
    )

    messages = [
        {
            "role": "system",
            "content": f"{SYSTEM_PROMPT}\n\n{INSTRUCOES_PERFIL[perfil_usuario]}"
        },
        {"role": "user", "content": instrucao_usuario},
    ]

    resposta = chamar_groq(messages, perfil_usuario=perfil_usuario)
    if not resposta:
        resposta = resposta_demo_sem_llm(pergunta, artigos, perfil_usuario=perfil_usuario)

    return f"**Perfil selecionado:** {perfil_usuario}\n\n" + resposta + montar_fontes(artigos)


print("✅ Funções criadas com perfis de usuário")


## 9. Testes automáticos do protótipo




In [ ]:
testes = [
    ("Dentro do escopo", "O que é Atendimento Educacional Especializado (AEE)?"),
    ("Dentro do escopo", "Como adaptar atividades para alunos com autismo?"),
    ("Dentro do escopo", "Como a tecnologia assistiva ajuda estudantes com deficiência visual?"),
    ("Fora do escopo", "Qual é a capital da França?"),
    ("Caso-limite", "Como ajudar uma criança?"),
]

for categoria, pergunta in testes:
    print("=" * 80)
    print("Categoria:", categoria)
    print("Pergunta:", pergunta)
    print("-" * 80)
    print(responder(pergunta)[:1200])
    print()

## 10. Interface de chat com Gradio


In [ ]:
import base64
from pathlib import Path

import gradio as gr


# ============================================================
# CONFIGURAÇÕES DO PROJETO
# ============================================================

PASTA_IMAGENS = Path("./imagens_do_tcc")

CAMINHO_LOGO_ACCENTURE = PASTA_IMAGENS / "logo_accenture.png"
CAMINHO_CORDAO = PASTA_IMAGENS / "CordaoTEA.svg"
CAMINHO_ICONE_GRUPO = PASTA_IMAGENS / "images.svg"
CAMINHO_CSS = Path("style.css")


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def carregar_imagem_base64(caminho: Path, tipo_mime: str) -> str:
    """
    Carrega uma imagem local e retorna uma URL base64 para usar no HTML/CSS.
    Exemplo de retorno: data:image/png;base64,...
    """
    if not caminho.exists():
        raise FileNotFoundError(
            f"Imagem não encontrada: {caminho}\n"
            "Verifique se a pasta 'imagens_do_tcc' está no mesmo local deste arquivo."
        )

    with caminho.open("rb") as arquivo:
        imagem_base64 = base64.b64encode(arquivo.read()).decode("utf-8")

    return f"data:{tipo_mime};base64,{imagem_base64}"


def carregar_css() -> str:
    """
    Lê o arquivo style.css e troca os marcadores pelas imagens em base64.
    Isso deixa o Python mais limpo e o CSS mais fácil de manter.
    """
    if not CAMINHO_CSS.exists():
        raise FileNotFoundError(
            "Arquivo style.css não encontrado.\n"
            "Deixe o notebook/app.py e o style.css na mesma pasta."
        )

    css = CAMINHO_CSS.read_text(encoding="utf-8")

    substituicoes = {
        "__CORDAO_TEA_URL__": CORDAO_TEA_URL,
        "__LOGO_ACCENTURE_URL__": LOGO_ACCENTURE_URL,
    }

    for marcador, valor in substituicoes.items():
        css = css.replace(marcador, valor)

    return css


# ============================================================
# IMAGENS DO FRONT
# ============================================================

LOGO_ACCENTURE_URL = carregar_imagem_base64(CAMINHO_LOGO_ACCENTURE, "image/png")
CORDAO_TEA_URL = carregar_imagem_base64(CAMINHO_CORDAO, "image/svg+xml")
ICONE_GRUPO_URL = carregar_imagem_base64(CAMINHO_ICONE_GRUPO, "image/svg+xml")


# ============================================================
# CSS PERSONALIZADO
# ============================================================

css_personalizado = carregar_css()


# ============================================================
# FUNÇÕES DO CHAT
# ============================================================

MENSAGEM_INICIAL_CHAT = [
    {
        "role": "assistant",
        "content": "Olá! Sou o Assistente Diversa. Como posso ajudar com educação inclusiva?"
    }
]

# CORREÇÃO PRINCIPAL:
# O erro "Token inesperado '<'" acontecia porque o backend estava devolvendo
# histórico no formato antigo de tuplas, mas a versão atual do Gradio espera
# mensagens no formato:
# [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]
#
# Por isso, abaixo o Chatbot trabalha sempre no formato "messages".


def criar_chatbot_compativel():
    """
    Cria o Chatbot usando o formato de mensagens.
    Funciona em versões recentes do Gradio.
    """
    parametros = {
        "label": "Assistente Diversa",
        "elem_id": "chatbot",
        "height": 490,
        "value": MENSAGEM_INICIAL_CHAT.copy(),
    }

    try:
        # Gradio 4/5
        return gr.Chatbot(**parametros, type="messages")
    except TypeError:
        # Gradio 6+
        return gr.Chatbot(**parametros)



def extrair_texto_conteudo_chat(conteudo):
    """
    Algumas versões do Gradio devolvem o conteúdo da mensagem como texto,
    e outras devolvem como lista de partes: [{"text": "...", "type": "text"}].
    Esta função normaliza tudo para texto puro.
    """
    if conteudo is None:
        return ""

    if isinstance(conteudo, str):
        return conteudo

    if isinstance(conteudo, list):
        textos = []
        for parte in conteudo:
            if isinstance(parte, dict):
                texto = parte.get("text") or parte.get("content") or ""
                if texto:
                    textos.append(str(texto))
            else:
                textos.append(str(parte))
        return "\n".join(textos)

    if isinstance(conteudo, dict):
        return str(conteudo.get("text") or conteudo.get("content") or "")

    return str(conteudo)


def normalizar_historico_chat(historico):
    """
    Garante que o histórico sempre fique no formato messages.
    """
    historico_messages = []

    if not historico:
        return historico_messages

    for item in historico:
        if isinstance(item, dict) and "role" in item and "content" in item:
            role = item.get("role", "assistant")
            if role not in ("user", "assistant"):
                role = "assistant"

            historico_messages.append({
                "role": role,
                "content": extrair_texto_conteudo_chat(item.get("content", ""))
            })

        elif isinstance(item, (tuple, list)) and len(item) == 2:
            pergunta_antiga, resposta_antiga = item

            if pergunta_antiga:
                historico_messages.append({
                    "role": "user",
                    "content": str(pergunta_antiga)
                })

            if resposta_antiga:
                historico_messages.append({
                    "role": "assistant",
                    "content": str(resposta_antiga)
                })

    return historico_messages


def enviar_mensagem(mensagem, historico, perfil_usuario="Professor"):
    """
    Envia a pergunta do usuário para a função responder(mensagem).
    Retorna sempre no formato correto do Gradio atual.
    """
    historico = normalizar_historico_chat(historico)

    if not mensagem or not mensagem.strip():
        return "", historico

    mensagem = mensagem.strip()

    try:
        resposta = responder(mensagem, historico, perfil_usuario)
    except NameError:
        resposta = (
            "A função responder(mensagem) ainda não foi configurada. "
            "Conecte aqui o back-end/RAG do projeto para gerar a resposta."
        )
    except Exception as erro:
        resposta = (
            "Não consegui gerar a resposta neste momento.\n\n"
            f"Detalhe técnico: {type(erro).__name__}: {erro}"
        )

    historico.append({"role": "user", "content": mensagem})
    historico.append({"role": "assistant", "content": str(resposta)})

    return "", historico


def limpar_chat():
    """Limpa o histórico da conversa."""
    return MENSAGEM_INICIAL_CHAT.copy()


def preencher_pergunta(texto):
    """Preenche o campo de pergunta ao clicar em uma sugestão."""
    return texto


# ============================================================
# TEMA DO GRADIO
# ============================================================

tema = gr.themes.Soft(
    primary_hue="purple",
    neutral_hue="slate",
    font=gr.themes.GoogleFont("Inter")
)


# ============================================================
# INTERFACE
# ============================================================

with gr.Blocks(
    theme=tema,
    css=css_personalizado,
    title="Assistente Diversa — Educação Inclusiva"
) as demo:

    # Cabeçalho: título do grupo e dropdown na mesma linha.
    # O visual fica no style.css para manter o Python limpo.
    with gr.Row(elem_id="topo-hero"):
        with gr.Column(scale=5, elem_id="coluna-hero"):
            gr.HTML(f"""
            <div id="hero-conteudo">
                <h1 class="titulo-grupo">
                    <img src="{ICONE_GRUPO_URL}" class="icone-grupo" alt="Ícone do grupo">
                    <span>Acessibilidade Inteligente</span>
                    <span class="detalhe-roxo">&gt;</span>
                </h1>

                <p class="subtitulo-equipe">Equipe Hélice</p>

                <div class="badges-hero">
                    <span class="badge-hero">Educação inclusiva</span>
                    <span class="badge-hero">Protótipo acadêmico</span>
                    <span class="badge-hero">Base com fontes reais</span>
                </div>
            </div>
            """)

        with gr.Column(scale=1, min_width=300, elem_id="box-perfil"):
            gr.HTML('<div class="perfil-label">Tipo de usuário</div>')

            # Dropdown de perfil: clique no campo para abrir e escolher entre as 3 opções.
            perfil_usuario = gr.Dropdown(
                choices=["Professor", "Família", "Gestor"],
                value="Professor",
                label=None,
                show_label=False,
                elem_id="perfil-rapido",
                interactive=True,
                allow_custom_value=False,
                filterable=False,
            )

    with gr.Row(elem_id="linha-principal"):

        with gr.Column(scale=4):
            chatbot = criar_chatbot_compativel()

            pergunta = gr.Textbox(
                label="Digite sua pergunta",
                placeholder="Exemplo: O que é AEE? Como incluir um aluno com TEA?",
                lines=3,
                elem_id="pergunta-textbox"
            )

            with gr.Row(elem_id="linha-botoes"):
                btn_enviar = gr.Button(
                    "Enviar pergunta",
                    elem_id="btn-enviar",
                    elem_classes=["botao-principal"],
                    variant="secondary"
                )
                btn_limpar = gr.Button(
                    "Limpar conversa",
                    elem_id="btn-limpar",
                    elem_classes=["botao-principal"],
                    variant="secondary"
                )

        with gr.Column(scale=1, elem_id="coluna-direita"):

            gr.HTML("""
            <div class="card-info">
                <h3>📚 Sobre o protótipo</h3>
                <p class="small-text">
                    Este assistente responde com base em uma base de conhecimento
                    cadastrada manualmente com fontes reais sobre educação inclusiva.
                </p>
                <p class="small-text">
                    Ele não substitui professores, profissionais do AEE, médicos,
                    psicólogos ou especialistas.
                </p>
            </div>
            """)

            with gr.Column(elem_id="card-sugestoes"):
                gr.HTML("""
                <h3>✅ Perguntas sugeridas</h3>
                <p class="small-text">
                    Clique em uma pergunta para preencher o campo de texto.
                </p>
                """)

                with gr.Column(elem_id="lista-sugestoes"):
                    btn_p1 = gr.Button(
                        "O que é educação inclusiva?",
                        elem_classes=["pergunta-btn"]
                    )
                    btn_p2 = gr.Button(
                        "O que é AEE?",
                        elem_classes=["pergunta-btn"]
                    )
                    btn_p3 = gr.Button(
                        "Como incluir aluno com TEA?",
                        elem_classes=["pergunta-btn"]
                    )

    gr.HTML("""
    <div class="footer">
        Projeto acadêmico desenvolvido para fins educacionais — SoulCode / Accenture
    </div>
    """)

    # ========================================================
    # EVENTOS DOS BOTÕES PRINCIPAIS
    # ========================================================

    btn_enviar.click(
        fn=enviar_mensagem,
        inputs=[pergunta, chatbot, perfil_usuario],
        outputs=[pergunta, chatbot]
    )

    pergunta.submit(
        fn=enviar_mensagem,
        inputs=[pergunta, chatbot, perfil_usuario],
        outputs=[pergunta, chatbot]
    )

    btn_limpar.click(
        fn=limpar_chat,
        inputs=[],
        outputs=[chatbot]
    )

    # ========================================================
    # EVENTOS DAS PERGUNTAS SUGERIDAS
    # ========================================================

    btn_p1.click(
        fn=lambda: preencher_pergunta("O que é educação inclusiva?"),
        inputs=[],
        outputs=[pergunta]
    )

    btn_p2.click(
        fn=lambda: preencher_pergunta("O que é Atendimento Educacional Especializado?"),
        inputs=[],
        outputs=[pergunta]
    )

    btn_p3.click(
        fn=lambda: preencher_pergunta("Como incluir um aluno com TEA em sala de aula?"),
        inputs=[],
        outputs=[pergunta]
    )


# ============================================================
# EXECUÇÃO DO APP
# ============================================================

if __name__ == "__main__":
    demo.launch(
        share=True,
        inbrowser=True,
        debug=True
    )
